In [ ]:
#Imports

import tensorflow as tf
import keras as k
import math
import pandas
import pandas as pd
import numpy
import numpy as np
import sklearn
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Dense

In [ ]:
# Carregando os dados

#Dataset Completo
dataset = pd.read_csv('johnson2.csv')

ano = pd.read_csv('johnson2.csv', usecols = [0])
revenue = pd.read_csv('johnson2.csv', usecols = [1])
revenue = revenue.values

dataset.head(10)

plt.plot(ano,revenue, label = 'lab revenue')
plt.xlabel('Quarter')
plt.ylabel('Revenue Johnson&Johnson')
plt.legend()
plt.show()

train_size = int(len(revenue) * 0.67)
test_size = len(revenue) - train_size
train, test = revenue[0:train_size,:], revenue[train_size:len(revenue),:]

def create_revenue(revenue, look_back = 1):
    dataX, dataY = [], []
    for i in range(len(revenue)-look_back-1):
        agt = revenue[i:(i+look_back), 0]
        dataX.append(agt)
        dataY.append(revenue[i + look_back, 0])
    return numpy.array(dataX), numpy.array(dataY)

# Reshape em X = t e Y = t + 1
look_back = 2
trainX, trainY = create_revenue(train, look_back)

testX, testY = create_revenue(test, look_back)

# Cria o modelo RNN com 1 input, 1 camada oculta com 32 neurônios e uma camada de saída
model = Sequential()

# Camada oculta
model.add( Dense(32, input_dim = look_back, activation = 'relu'))

# Camada de saída
model.add(Dense(1))

# Compilação
model.compile(optimizer = 'adam',loss = 'mean_squared_error' )

# Fit do modelo
model.fit(trainX, trainY, epochs = 500, batch_size = 2, verbose = 2)

trainScore = model.evaluate(trainX, trainY, verbose = 0)

print('Score em Treino: %.2f MSE (%.2f RMSE)' % (trainScore, math.sqrt(trainScore)))

testScore = model.evaluate(testX, testY, verbose = 0)

print('Score em Teste: %.2f MSE (%.3f RMSE)' % (testScore, math.sqrt(testScore)))


# Gera previsões
trainPredict = model.predict(trainX)
testPredict = model.predict(testX)

# Ajusta os dados de treino para o Plot
trainPredictPlot = numpy.empty_like(revenue)
trainPredictPlot[:, :] = numpy.nan
trainPredictPlot[look_back:len(trainPredict)+look_back, :] = trainPredict

# Ajusta os dados de teste para o Plot
testPredictPlot = numpy.empty_like(revenue)
testPredictPlot[:, :] = numpy.nan
testPredictPlot[len(trainPredict)+(look_back*2)+1:len(revenue)-1, :] = testPredict


plt.plot(revenue, label = 'Dados reais', color = 'b')
plt.plot(trainPredictPlot, label = "Predição da rede neural",color = 'r')
plt.plot(testPredictPlot, color = 'r')
plt.xlabel('Quarter')
plt.ylabel('Revenue')
plt.legend()
plt.show()

#Previsão do Modelo - FORECASTING
y_pred = model.predict(testX[-1].reshape(1,-1))
print(y_pred)

plt.plot(revenue, label = 'Real', color = 'b')
plt.plot(trainPredictPlot, label = "Predito pela RN",color = 'r')
plt.scatter(74,y_pred,color='black',label = 'forcasting')
plt.plot(testPredictPlot, color = 'r')
plt.xlabel('Quarter')
plt.xticks(np.arange(0, 85, 15))
plt.ylabel('Revenue')
plt.legend()
plt.show()

#Análise de desempenho do modelo
mse_treino = sklearn.metrics.mean_squared_error(trainPredict, trainY)
print('MSE_treino score: %0.3f' % mse_treino)
print('')
rmse_treino = math.sqrt(mse_treino)
print('RMSE_treino score: %0.3f' % rmse_treino)

print('')

mse_teste = sklearn.metrics.mean_squared_error(testPredict, testY)
print('RMSE_treino score: %0.3f' %mse_teste)
print('')
rmse_teste = math.sqrt(mse_teste)
print('RMSE_teste score: %0.3f'%rmse_teste)

R2_treino = r2_score(trainPredict,trainY)
print ('Coeficiente Treino R² score: %0.3f' %  R2_treino)

R2_teste = r2_score(testPredict,testY)
print ('Coeficiente Teste R²score: %0.3f' %  R2_teste)

MAE_treino = mean_absolute_error(trainPredict,trainY)
print ('MAE Train score: %0.3f' %  MAE_treino)

MAE_test = mean_absolute_error(testPredict,testY)
print ('MAE Test score: %0.3f' %  MAE_test)

# Tabela de Resultados
RNN_Treino = {'MSE': mse_treino,
       'RMSE':rmse_treino,
       'R²':R2_treino,
       'MAE':MAE_treino}

RNN_Teste = {'MSE': mse_teste,
             'RMSE':rmse_teste,
             'R²': R2_teste,
             'MAE':MAE_test}

# Concatena todos os dicionários em um dataframe do Pandas
resumo = pd.DataFrame({'DESEMPENHO RNN TREINO':pd.Series(RNN_Treino),'DESEMPENHO RNN TESTE':pd.Series(RNN_Teste)})
# Print
resumo

# Compilação Treino com MAE
model.compile(optimizer = 'adam',loss= 'mean_absolute_error' )

history2 = model.fit(trainX, trainY, epochs = 500, batch_size = 2, verbose = 2)

# Compilação Teste com MAE
model.compile(optimizer = 'adam',loss= 'mean_absolute_error' )

history3 = model.fit(testX, testY, epochs = 500, batch_size = 20, verbose = 2)

# Compilação Treino com MAE
model.compile(optimizer = 'adam',loss= 'mean_absolute_error' )

history2 = model.fit(trainX, trainY, epochs = 500, batch_size = 2, verbose = 2)

# Compilação Teste com MAE
model.compile(optimizer = 'adam',loss= 'mean_absolute_error' )

history3 = model.fit(testX, testY, epochs = 500, batch_size = 20, verbose = 2)

plt.plot(history2.history['loss'])
plt.plot(history3.history['loss'])
plt.title('Evolução do MAE durante etapas')
plt.ylabel('MAE')
plt.xlabel('Épocas')
plt.legend(['MAE durante treino', 'MAE durante teste '], loc='upper left')
plt.show()